In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
df = pd.read_pickle("SCOP_taxonomy.pkl")
df

,fold,taxid,taxonomy
9,a.1.1.1,1423,cellular organisms Bacteria Terrabacteria gr...
11,a.1.1.1,5885,cellular organisms Eukaryota Sar Alveolata ...
14,a.1.1.1,3054,cellular organisms Eukaryota Viridiplantae ...
17,a.1.1.1,1773,cellular organisms Bacteria Terrabacteria gr...
34,a.1.1.1,1773,cellular organisms Bacteria Terrabacteria gr...
...,...,...,...
362200,k.13.1.1,1280,cellular organisms Bacteria Terrabacteria gr...
362462,k.37.1.1,32630,other entries other sequences artificial seq...
362467,k.37.1.1,32644,unclassified entries unclassified sequences
362475,k.37.1.1,32630,other entries other sequences artificial seq...


In [3]:
df.loc[df['taxonomy'].str.contains("Bacteria"),  "taxonomy"] = "Bacteria"
df.loc[df['taxonomy'].str.contains("Archaea"),   "taxonomy"] = "Archaea"
df.loc[df['taxonomy'].str.contains("Eukaryota"), "taxonomy"] = "Eukaryota"
df

,fold,taxid,taxonomy
9,a.1.1.1,1423,Bacteria
11,a.1.1.1,5885,Eukaryota
14,a.1.1.1,3054,Eukaryota
17,a.1.1.1,1773,Bacteria
34,a.1.1.1,1773,Bacteria
...,...,...,...
362200,k.13.1.1,1280,Bacteria
362462,k.37.1.1,32630,other entries other sequences artificial seq...
362467,k.37.1.1,32644,unclassified entries unclassified sequences
362475,k.37.1.1,32630,other entries other sequences artificial seq...


In [4]:
df = df[df['taxonomy'].isin(["Bacteria","Archaea","Eukaryota"])]
df

,fold,taxid,taxonomy
9,a.1.1.1,1423,Bacteria
11,a.1.1.1,5885,Eukaryota
14,a.1.1.1,3054,Eukaryota
17,a.1.1.1,1773,Bacteria
34,a.1.1.1,1773,Bacteria
...,...,...,...
361904,j.150.1.1,4932,Eukaryota
361944,k.1.1.1,44689,Eukaryota
361948,k.1.1.1,4513,Eukaryota
362188,k.12.1.1,10090,Eukaryota


In [5]:
#keep proteins in the first 7 SCOPe classes
df = df[df['fold'].str[0].isin(['a','b','c','d','e','f','g'])]
df

,fold,taxid,taxonomy
9,a.1.1.1,1423,Bacteria
11,a.1.1.1,5885,Eukaryota
14,a.1.1.1,3054,Eukaryota
17,a.1.1.1,1773,Bacteria
34,a.1.1.1,1773,Bacteria
...,...,...,...
353272,g.101.1.1,4932,Eukaryota
353275,g.101.1.1,9606,Eukaryota
353285,g.102.1.1,9606,Eukaryota
353302,g.102.1.0,9606,Eukaryota


In [6]:
all_sp_lines = df.index.astype('int')
fold = []
dom  = []
sp   = []
boundary = []
with open("./data/dir.des.scope.2.08-stable.txt","r") as f:
    i = 0
    for line_no, line in enumerate(f):
        if line_no==all_sp_lines[i]:
            species = df.loc[str(line_no)]['taxid']
            i+=1
        line = line.split()
        if line[1] == "px":

            fold.append(line[2])
            dom.append(line[3])
            boundary.append("".join(line[4:]))
            sp.append(species)            
        if i>=len(all_sp_lines):
            break

all_domain_sp = pd.DataFrame ({"fold": fold, "domain": dom, "taxid":sp, "boundary":boundary})
all_domain_sp['PDB'] = all_domain_sp['domain'].str[1:5].str.upper() + "_" + all_domain_sp['domain'].str[5].str.upper()
all_domain_sp

,fold,domain,taxid,boundary,PDB
0,a.1.1.1,d1ux8a_,1423,1ux8A:,1UX8_A
1,a.1.1.1,d1dlwa_,5885,1dlwA:,1DLW_A
2,a.1.1.1,d1uvya_,5885,1uvyA:,1UVY_A
3,a.1.1.1,d1dlya_,3054,1dlyA:,1DLY_A
4,a.1.1.1,d1uvxa_,3054,1uvxA:,1UVX_A
...,...,...,...,...,...
302596,g.102.1.0,d5fz0a2,9606,5fz0A:623-753,5FZ0_A
302597,g.102.1.0,d5fy5a2,9606,5fy5A:623-753,5FY5_A
302598,g.102.1.0,d5fzma2,9606,5fzmA:623-753,5FZM_A
302599,g.102.1.0,d5fz3a2,9606,5fz3A:623-753,5FZ3_A


### Filter for PDB chains in pdb70, check for sequence identity with training data.

In [7]:
with open("./data/pdb70_a3m.FFINDEX", "r") as f:
     pdb70 = [x.split()[0] for x in f.readlines()]

all_domain_sp = all_domain_sp[all_domain_sp['PDB'].isin(pdb70)]
all_domain_sp

,fold,domain,taxid,boundary,PDB
0,a.1.1.1,d1ux8a_,1423,1ux8A:,1UX8_A
1,a.1.1.1,d1dlwa_,5885,1dlwA:,1DLW_A
3,a.1.1.1,d1dlya_,3054,1dlyA:,1DLY_A
4,a.1.1.1,d1uvxa_,3054,1uvxA:,1UVX_A
18,a.1.1.1,d1s56b_,1773,1s56B:,1S56_B
...,...,...,...,...,...
302539,g.101.1.1,d2k0aa1,4932,2k0aA:2-107,2K0A_A
302541,g.101.1.1,d5syba_,9606,5sybA:,5SYB_A
302547,g.102.1.1,d6ek6a2,9606,6ek6A:623-754,6EK6_A
302560,g.102.1.0,d5fyza2,9606,5fyzA:623-753,5FYZ_A


### Export this list of PDB ID's to then filter out any sequences with similarity to training data, and pairwise sequence identity from PISCES web server

In [ ]:
with open("./data/taxonomy_export_list.txt","w") as f:
    for x in all_domain_sp['PDB']:
        f.write(x + "\n")

### After filtering the list of PDB chains, import filtered list and use subset of dataframe

In [9]:
with open("./data/taxonomy_import_list.txt", "r") as f:
    keep_chains = f.readlines()

all_domain_sp = all_domain_sp[all_domain_sp['PDB'].isin(keep_chains)]
all_domain_sp

,fold,domain,taxid,boundary,PDB
0,a.1.1.1,d1ux8a_,1423,1ux8A:,1UX8_A
1,a.1.1.1,d1dlwa_,5885,1dlwA:,1DLW_A
3,a.1.1.1,d1dlya_,3054,1dlyA:,1DLY_A
4,a.1.1.1,d1uvxa_,3054,1uvxA:,1UVX_A
18,a.1.1.1,d1s56b_,1773,1s56B:,1S56_B
...,...,...,...,...,...
302448,g.96.1.0,d6urvc_,9606,6urvC:,6URV_C
302503,g.97.1.1,d3fnvb_,9606,3fnvB:,3FNV_B
302533,g.98.1.1,d2rf4f_,4932,2rf4F:,2RF4_F
302539,g.101.1.1,d2k0aa1,4932,2k0aA:2-107,2K0A_A


In [10]:
taxdomain   = dict(zip(df.taxid,df.taxonomy))
all_domain_sp['tax_domain'] = [taxdomain[x] for x in all_domain_sp['taxid']]
all_domain_sp

C:\Users\Mohan2\AppData\Local\Temp\ipykernel_8380\2027787187.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_domain_sp['tax_domain'] = [taxdomain[x] for x in all_domain_sp['taxid']]


,fold,domain,taxid,boundary,PDB,tax_domain
0,a.1.1.1,d1ux8a_,1423,1ux8A:,1UX8_A,Bacteria
1,a.1.1.1,d1dlwa_,5885,1dlwA:,1DLW_A,Eukaryota
3,a.1.1.1,d1dlya_,3054,1dlyA:,1DLY_A,Eukaryota
4,a.1.1.1,d1uvxa_,3054,1uvxA:,1UVX_A,Eukaryota
18,a.1.1.1,d1s56b_,1773,1s56B:,1S56_B,Bacteria
...,...,...,...,...,...,...
302448,g.96.1.0,d6urvc_,9606,6urvC:,6URV_C,Eukaryota
302503,g.97.1.1,d3fnvb_,9606,3fnvB:,3FNV_B,Eukaryota
302533,g.98.1.1,d2rf4f_,4932,2rf4F:,2RF4_F,Eukaryota
302539,g.101.1.1,d2k0aa1,4932,2k0aA:2-107,2K0A_A,Eukaryota


In [12]:
keep_folds = []
keep_fold2 = []

for family in df['fold'].unique():
    tax_entries = df[df['fold']==family]['taxonomy'].unique()
    if 'Bacteria' in tax_entries:
        if 'Eukaryota' in tax_entries:
            keep_folds.append(family)
            if 'Archaea' in tax_entries:
                keep_fold2.append(family)

In [16]:
all_domain_sp[all_domain_sp['fold'].isin(keep_fold2)].to_pickle("taxonomy.pkl")
all_domain_sp[all_domain_sp['fold'].isin(keep_folds)].to_pickle("taxonomy2.pkl")